# Action client node

## Imports

In [1]:
from assignment_2_2024.msg import PlanningAction, PlanningGoal

import rospy
import actionlib

from nav_msgs.msg import Odometry
from ass2_ros1.msg import RobotVelocity


## Globals


In [2]:
client = None
goal = None
latest_feedback = None

new_goal = False
target_reached = False

vel_pub = None

In [3]:
def read_feedback(feedback):
    global target_reached, new_goal, latest_feedback
    latest_feedback = feedback;

    if (new_goal and feedback.stat == 'Target reached!'):
        rospy.print("Target reached !")
        target_reached = True
        new_goal = False;

In [4]:
def publish_robot_velocity(msg):
    vel = RobotVelocity()
    vel.x = msg.pose.pose.position.x
    vel.y = msg.pose.pose.position.y
    vel.vel_x = msg.twist.twist.linear.x
    vel.vel_z = msg.twist.twist.angular.z

    vel_pub.publish(vel)

In [5]:
def update_goal():
    global goal
    goal = PlanningGoal()
    goal.target_pose.header.frame_id = "map"
    #TODO: inputcheck 
    goal.target_pose.pose.position.x = float(input("insert the x for the goal: "))
    goal.target_pose.pose.position.y = float(input("insert the y for the goal: "))
    client.send_goal(goal, feedback_cb=read_feedback)

In [6]:
def print_status():
    if (not goal):
        print("No goal set yet")
        return 
    
    print(goal)
    
    if (not latest_feedback):
        print("No feedacks received from the server yet!")
        return
    print(latest_feedback)

In [7]:
def get_choiche(prompt, choice_list):
    user_input = 0
    input_ok = False

    while not input_ok:
        print(prompt)
        for i, choice in enumerate(choice_list):
            print(f"{i+1}. {choice}")
        
        try:
            user_input = int(input("> ")) - 1
            if user_input >=0 and user_input < len(choice_list):
                input_ok = True
        except:
            pass
        else:
            input_ok = True
        
        if not input_ok:
            print(f"Please input a valid integer between {1} and {len(choice_list)}!")
    
    return user_input

In [ ]:
rospy.sleep(2)
global client, vel_pub

rospy.init_node('action_client')

vel_pub = rospy.Publisher("/robot_status", RobotVelocity, queue_size = 10)
rospy.Subscriber("/odom", Odometry, publish_robot_velocity)

client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)

choices = {"New/Update goal": update_goal, "Get status": print_status, "Quit": exit}
choice_list = list(choices.keys())

rate = rospy.Rate(10)
while True:
    choice = get_choiche("Select what you want to do: ", choice_list)
    choices[choice_list[choice]]()
    rate.sleep()


Select what you want to do: 
1. New/Update goal
2. Get status
3. Quit
> 1
insert the x for the goal: 2
insert the y for the goal: 2
Select what you want to do: 
1. New/Update goal
2. Get status
3. Quit
> 2
target_pose: 
  header: 
    seq: 0
    stamp: 
      secs: 0
      nsecs:         0
    frame_id: "map"
  pose: 
    position: 
      x: 2.0
      y: 2.0
      z: 0.0
    orientation: 
      x: 0.0
      y: 0.0
      z: 0.0
      w: 0.0
actual_pose: 
  position: 
    x: -0.011919930658763055
    y: 1.00727085907254
    z: 0.10000234987003985
  orientation: 
    x: -4.9245250517701016e-06
    y: 2.9159921380083176e-06
    z: 0.07071605305777988
    w: 0.9974964861126958
stat: "State 1: avoid obstacle"
Select what you want to do: 
1. New/Update goal
2. Get status
3. Quit
